### Zoteroize and Obsidianize a `Save my Chatbot` Perplexity Dialogue

Replace the citation numbers in a saved Save my Chatbot Perplexity dialogue with matching literature note or zotero item links

In [1]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict, Counter
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic
import re
from typing import Optional, Dict, List, Tuple

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [2]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

savemychatbot_perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_multi_prompt_savemychatbot_example.md'
#savemychatbot_perplexity_dialog_file = pl.Path(r"C:\Users\scott\tmp\Second 2025-02-03_11-37-09_Perplexity.ai_Loading a zotero database using pyzotero is very slow. What....md")
output_file_savemychatbot = tmp_dir / 'tmp_savemychatbot_multiprompt_perplexity_example.md'

In [3]:
TOP_HEADING_LEVEL_IN_AI = 3
MIN_SCORE_TITLE_MATCH = 95 # max==100: stringent, limit false matches

##### get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes


In [4]:
zotero_cache = rfw.ZoteroCache()
parentItems = zotero_cache.get_data()

Cache is valid. Reading data from cache.


In [6]:
# Collect info about each zotero DB item that has a URL
lit_note_file_stems = {fNm.stem for fNm in rfw.lit_notes_obsidian_dir.glob('*.md')}

zot_db_items = []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    if not (title := pdat.get('title')):
        continue

    citekeyThis = rfw.get_citation_key(pdat)
    zot_db_items.append(dict(citekey=citekeyThis, zotkey=parent['key'], title=title, hasLitNote=citekeyThis in lit_note_file_stems))

    if url := pdat.get('url'):
        if normalized_url :=rfw.normalize_url(url):
            citekeysForURL[normalized_url].append(citekeyThis)

if repeatedURLs := {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}:
    print(f"Found {len(repeatedURLs)} URLs with > 1 parent (citekey)")
    for url, citekeys in repeatedURLs.items():
        print(f"{', '.join(citekeys)}\n\t{url}")
    raise Exception(f'Not written for repeated URLs')

citekey_to_url = {citekeys[0]: url for url, citekeys in citekeysForURL.items()}
url_to_citekey = {url: citekey for citekey, url in citekey_to_url.items()}

zot_db_items = pd.DataFrame(zot_db_items).set_index('citekey')
zot_db_items['url'] = pd.Series(citekey_to_url)
zot_db_items = zot_db_items.reset_index()

if sum(hasNoURL := zot_db_items.url.isna()):
    print(f"Dropping {sum(hasNoURL)} of {len(zot_db_items)} zotero entries with no URL:")
    display((zot_db_items_no_url := zot_db_items[hasNoURL]).head())
    zot_db_items = zot_db_items[~hasNoURL]


Dropping 149 of 1682 zotero entries with no URL:


,citekey,zotkey,title,hasLitNote,url
2,MMSDataModelSummary_v5.2,8XJHRYMU,MMS Data Model Package Summary v5.2,False,NaN
17,Seals99irradFrcstDiag,WYP9J7EU,The heart of suny irradiance forecasting,False,NaN
93,LaPaglia13TestIncrsSuggestibility,LDTF7M3L,Testing increases suggestibility for narrative...,False,NaN
154,Gaur20attribModellingRvw,HLHKVCLX,Attribution modelling in marketing: Literature...,False,NaN
201,Wang19predOptcoolLdFrcst,R7TJLE7Y,Cooling load forecasting-based predictive opti...,False,NaN


## For markdown from the "Save my Chatbot" chrome/firefox extension

In [7]:
def find_zotero_item_by_url(url: str, zot_db_items: pd.DataFrame) -> Optional[Dict]:
    """Find a Zotero item by its URL."""
    normalized_url = rfw.normalize_url(url)
    matches = zot_db_items[zot_db_items['url'] == normalized_url]
    if not matches.empty:
        return matches.iloc[0].to_dict()  # Return as a dictionary
    return None

def find_zotero_item_by_title(target_title: str, zot_db_items: pd.DataFrame) -> Optional[Dict]:
    """Find the Zotero item with the best matching title."""
    zotero_items = zot_db_items.to_dict('records')
    best_match_item = None
    best_score = 0

    for item in zotero_items:
        score = rfw.match_titles(target_title, item['title'], main_title_only=False)
        if score > best_score:
            best_match_item = item.copy()
            best_score = score

    if best_score > MIN_SCORE_TITLE_MATCH:
        return best_match_item 
    
    return None

def build_source_url_to_title(sources_content: str) -> Dict[str, str]:
    """Build a dictionary mapping URLs to titles from the sources section."""
    # if sources_content is None: # if source doc had no Sources section
    #     return {} 
    
    source_url_to_title = {}
    matches = re.findall(r'- \[(.*?)\]\((https?://\S+)\)', sources_content)
    for title, url in matches:
        normalized_url = rfw.normalize_url(url)
        title = re.sub(r'^\s*\(\d+\)\s*', '', title) # remove ref num
        source_url_to_title[normalized_url] = title.strip()
    return source_url_to_title

def replace_links_with_zotero_items(
    body_content: str,
    sources_content: str,
    zot_db_items: pd.DataFrame,
) -> Tuple[str, str, Counter]:
    """
    Replace links in body content and sources content with Zotero links or leave them as-is.
    
    Returns:
        - Updated body content.
        - Updated sources content.
        - A Counter of URLs in the body that were not found in the sources."""
    
    source_url_to_title = build_source_url_to_title(sources_content)

    unsourced_body_links = Counter()
    body_link_num_not_in_zotero = {}

    def link_to_obsidian_or_zotero(zotero_item):
        """Returns link to Obsidian lit note if it exists, else to zotero item"""
        if zotero_item.get('hasLitNote', False):
            obsidian_citekey = zotero_item["citekey"]
            return f'[[{obsidian_citekey}|{obsidian_citekey}]]'
        else:
            link_text = f'{zotero_item["citekey"]}\u2794{zotero_item["zotkey"]}'
            return rfw.zotero_item_link(zotero_item["zotkey"], link_text)

    def make_my_lit_link(url):
        """If a zotero item has a matching url, or title that matches a source's 
        section link title, then return a link to that item or its obsidian note."""

        if zotero_item := find_zotero_item_by_url(url, zot_db_items):
            return link_to_obsidian_or_zotero(zotero_item)

        if url in source_url_to_title:
            # try to replace matching source link title with zotero item title
            title = source_url_to_title[url]
            if zotero_item := find_zotero_item_by_title(title, zot_db_items):
                return link_to_obsidian_or_zotero(zotero_item)

        return None # no kind of zotero item match
            
    def swap_my_lit_link_body(doc_match):
        """Replace a body section link with one pointing to zotero/obsidian, if possible.
        Otherwise highlight it so it's clear there was no match
        
        Arg: doc_match: a regexp match object to a documenent body section link"""
         
        url_body_link = rfw.normalize_url(doc_match.group(2))
        body_link_num = doc_match.group(1)
        
        if url_body_link not in source_url_to_title:
            unsourced_body_links[url_body_link] += 1 # for later error reporting

        if my_link := make_my_lit_link(url_body_link):
            return my_link

        # a link in body that wasn't in zotero: highlight it in both the body and the sources
        body_link_num_not_in_zotero[body_link_num] = True
        return f'=={doc_match.group(0)}=='

    def append_my_lit_link_source(doc_match):
        """Append a sources section link with a highlighted link pointing to zotero/obsidian, if possible.
        
        Arg: doc_match: a regexp match object to a document sources section link"""

        url_source_link = rfw.normalize_url(doc_match.group(3))

        source_link_num = doc_match.group(1)
        output_link_num = f'({source_link_num})'
        descript_source_link = doc_match.group(2)
        if my_link := make_my_lit_link(url_source_link):
            return f'[{output_link_num} {descript_source_link}]({url_source_link}) **{my_link}**'

        if body_link_num_not_in_zotero.get(source_link_num):
            output_link_num = f'=={output_link_num}==' # highlight it, to match body appearance
        
        return f'[{output_link_num} {descript_source_link}]({url_source_link})'
    
    relinked_body_content = re.sub(r'\[(.*?)\]\((https?://\S+)\)', swap_my_lit_link_body, body_content)
    relinked_body_content = rfw.setext_headers_to_atx(relinked_body_content, TOP_HEADING_LEVEL_IN_AI)

    relinked_sources_content = re.sub(r'\[\((\d+)\)\s*(.*?)\]\((https?://\S+)\)', append_my_lit_link_source, sources_content)
    return relinked_body_content, relinked_sources_content, unsourced_body_links

def capitalize_first_word_if_needed(text):
    # Check if the first word already contains any capital letters
    first_word = text.split()[0] if text.strip() else ""
    if any(char.isupper() for char in first_word):
        return text  # Return the original string if the first word has capitals
    else:
        return text[0].upper() + text[1:] if text else text
    
def get_first_m_words(text, m, stop_phrase=None):

    if stop_phrase and stop_phrase in text:
        text = text.split(stop_phrase)[0]
    
    words = text.split()
    return ' '.join(words[:m])

def summarize_prompt(prompt, num_words, stop_phrase=None):
    prompt = get_first_m_words(prompt, num_words, stop_phrase)
    return rfw.convert_to_atx_header(f'User: "{capitalize_first_word_if_needed(prompt)}..."', TOP_HEADING_LEVEL_IN_AI - 1)

def relink_perplexity_export_SmC(input_file: str, output_file: str, zot_db_items: pd.DataFrame):
    """Process a markdown file to replace links with Zotero references."""
    
    with open(input_file, 'r') as infile:
        content = infile.read()

    # Split content into sections based on level 2 headers named "User"
    sections = re.split(r'(?<=\n)## User', content)

    processed_sections = [sections[0]]  # start with header
    log_missing_links = []

    for section_idx, section in enumerate(sections[1:], start=1):  # Skip anything before the first "User" section
        # Split each section into body and sources parts
        parts = re.split(r'(\n---\s*\n\s*\*\*Sources:\*\*\s*\n)', section)
        
        if len(parts) < 3:
            print('Incomplete Body/Sources pair: assume no Sources for this section')
            parts.append("")
            parts.append("")
        else:
            parts[1] = f"\n{rfw.convert_to_atx_header('Sources', TOP_HEADING_LEVEL_IN_AI)}\n"

        body_content = parts[0]
        
        # Replace links in body and sources parts with Zotero or Obsidian references
        updated_body_content, updated_sources_content, unsourced_body_links_counter = replace_links_with_zotero_items(
            body_content,
            parts[2],
            zot_db_items,
        )

        if unsourced_body_links_counter:
            log_missing_links.append(
                f"Section {section_idx}: Body links not in source: " +
                ", ".join([f"{url} (count: {count})" for url, count in unsourced_body_links_counter.items()])
            )

        # clue = get_first_m_words(updated_body_content, 10, "## AI answer")
        # processed_sections.append(f'## User: "{clue}"{updated_body_content}')

        short_prompt = summarize_prompt(updated_body_content, 10, "## AI answer")
        processed_sections.append(short_prompt)
        processed_sections.append(updated_body_content)
        processed_sections.append(parts[1]) # Sources title text

        processed_sections.append(updated_sources_content)

    with open(output_file, 'w') as outfile:
        outfile.write(''.join(processed_sections))

    if log_missing_links:
        print("Log of missing links:")
        for log_entry in log_missing_links:
            print(log_entry)

In [8]:
ic(savemychatbot_perplexity_dialog_file, output_file_savemychatbot)
relink_perplexity_export_SmC(savemychatbot_perplexity_dialog_file, output_file_savemychatbot, zot_db_items)
print('Done.')

ic| savemychatbot_perplexity_dialog_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_multi_prompt_savemychatbot_example.md')
    output_file_savemychatbot: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_savemychatbot_multiprompt_perplexity_example.md')


Log of missing links:
Section 3: Body links not in source: https://en.wikipedia.org/wiki/total_correlation (count: 4), https://arxiv.org/abs/2011.04794 (count: 3), https://proceedings.mlr.press/v206/bai23a/bai23a.pdf (count: 4)
Done.
